# YouTube Video Transcription

This notebook automates the process of checking availability, downloading audio, transcribing with Whisper, and cleaning up audio files.

**Pipeline**: Availability check → Audio download (yt-dlp) → Transcription (Whisper) → Compression and cleanup

In [ ]:
import pandas as pd
import warnings
import requests
import os
import subprocess
import whisper
import shutil
from tqdm.notebook import tqdm
import multiprocessing as mp
from itertools import cycle
import torch

warnings.filterwarnings('ignore')

In [ ]:
# Define generic project paths (use repository root as base)
PROJECT_ROOT = os.getcwd()
RAW_DATA_DIR = os.path.join(PROJECT_ROOT, 'raw_data')
CLEANED_DATA_DIR = os.path.join(PROJECT_ROOT, 'cleaned_data')
LOG_DIR = os.path.join(PROJECT_ROOT, 'filtrated_data_log')
DOWNLOAD_LOG_DIR = os.path.join(PROJECT_ROOT, 'download_log')
AUDIO_DIR = os.path.join(PROJECT_ROOT, 'audios')
TRANSCRIPTION_DIR = os.path.join(PROJECT_ROOT, 'transcriptions')

# Ensure required folders exist
for folder in [CLEANED_DATA_DIR, LOG_DIR, DOWNLOAD_LOG_DIR, AUDIO_DIR, TRANSCRIPTION_DIR]:
    os.makedirs(folder, exist_ok=True)

print('Project folders are set up under', PROJECT_ROOT)

In [ ]:
# df = pd.read_csv(os.path.join(PROJECT_ROOT, 'cleaned_data', 'final_comments_with_sentiment.csv'))
df = pd.read_csv(os.path.join(CLEANED_DATA_DIR, 'final_comments_with_sentiment.csv'))

In [ ]:
df['video_id'].nunique()

---
## 1. Video Availability Check

This section checks which videos in the dataset are still available on YouTube.

The first step is to filter the videos in our dataset to determine which are still available on YouTube.

In [ ]:
def check_video_availability(video_id):
    url = f"https://www.youtube.com/oembed?url=https://www.youtube.com/watch?v={video_id}&format=json"
    response = requests.get(url)
    
    return response.status_code == 200

In [ ]:
def save_progress(valid_videos, invalid_videos):
    """Save valid and invalid videos to CSV files under the project log folder."""
    valid_videos.to_csv(os.path.join(LOG_DIR, 'valid_videos.csv'), index=False)
    invalid_videos.to_csv(os.path.join(LOG_DIR, 'invalid_videos.csv'), index=False)

In [ ]:
def load_progress():
    """Load progress of valid and invalid videos (if present)."""
    try:
        valid_videos = pd.read_csv(os.path.join(LOG_DIR, 'valid_videos.csv'))
    except FileNotFoundError:
        valid_videos = pd.DataFrame(columns=["video_id"])

    try:
        invalid_videos = pd.read_csv(os.path.join(LOG_DIR, 'invalid_videos.csv'))
    except FileNotFoundError:
        invalid_videos = pd.DataFrame(columns=["video_id"])

    return valid_videos, invalid_videos

In [ ]:
valid_videos, invalid_videos = load_progress()

In [ ]:
for video_id in tqdm(df['video_id'].drop_duplicates(), desc="Verificando disponibilidade de vídeos"):
    if video_id not in valid_videos["video_id"].values and video_id not in invalid_videos["video_id"].values:
        status = check_video_availability(video_id)
        
        if status is not None:
            new_row = pd.DataFrame({"video_id": [video_id]})
            if status:
                valid_videos = pd.concat([valid_videos, new_row], ignore_index=True)
                #print(f"O vídeo {video_id} está disponível.")
            else:
                invalid_videos = pd.concat([invalid_videos, new_row], ignore_index=True)
               # print(f"O vídeo {video_id} foi removido ou está indisponível.")
        
        save_progress(valid_videos, invalid_videos)

---
## 2. Dataset Filtering

Filters the original dataset keeping only comments from videos that are still available. This ensures only accessible content is processed in subsequent steps.

In [ ]:
df_filtro = pd.read_csv(os.path.join(LOG_DIR, 'valid_videos.csv'))

In [ ]:
df['video_id'] = df['video_id'].astype(str)
df_filtro['video_id'] = df_filtro['video_id'].astype(str)

In [ ]:
dados_filtrados = df[df['video_id'].isin(df_filtro['video_id'])]

In [ ]:
dados_filtrados.head()

---
## 3. Audio Download

Downloads audio tracks from available videos using **yt-dlp**:
- **Format**: MP3 (best available audio quality)
- **Cookies**: Uses a cookies file to bypass restrictions when needed
- **Progress**: Logging prevents re-downloading already processed videos
- **Error handling**: Captures and logs download failures

In [ ]:
# Configuration
audio_folder = AUDIO_DIR
# log_file = ".../download_log/download_progress.log"
log_file = os.path.join(DOWNLOAD_LOG_DIR, 'download_progress.log')
# csv_file = path to valid videos CSV
csv_file = os.path.join(LOG_DIR, 'valid_videos.csv')

In [ ]:
def load_progress():
    if os.path.exists(log_file):
        with open(log_file, "r") as f:
            return set(f.read().splitlines())
    return set()

def save_progress(video_id):
    with open(log_file, "a") as f:
        f.write(video_id + "\n")

In [ ]:
df = pd.read_csv(csv_file)
video_ids = df["video_id"].astype(str).tolist()

In [ ]:
completed_videos = load_progress()

In [ ]:
cookies = 'cookies_1306_02.txt'

for video_id in tqdm(video_ids, desc="Downloading audios"):
    audio_path = os.path.join(audio_folder, f"{video_id}.mp3")

    if video_id in completed_videos or os.path.exists(audio_path):
        continue

    url = f"https://www.youtube.com/watch?v={video_id}"
    print(f"Downloading audio from {url}...")

    try:
        command = [
            "yt-dlp",
            "--cookies", cookies,
            "-f", "bestaudio",
            "--extract-audio",
            "--audio-format", "mp3",
            "-o", f"{audio_folder}/%(id)s.%(ext)s",
            url
        ]
        
        result = subprocess.run(
            command,
            check=True, text=True, capture_output=True
        )

        print(f"Download completed: {audio_path}")
        save_progress(video_id)

    except subprocess.CalledProcessError as e:
        print(f"Error downloading {video_id}: {e.stderr}")

print("Download process finished!")

---
## 4. Audio Transcription

Transcribes downloaded audio using the **Whisper** model (OpenAI):
- **Model**: small (run on GPU via CUDA for better performance)
- **Language**: Portuguese
- **Output**: Text files (.txt) with full transcription
- **Optimization**: After transcription, audio is compressed (.zip) and original removed to save space
- **Checkpointing**: Avoids retranscribing already processed videos

In [ ]:
# audio_folder = AUDIO_DIR
audio_folder = AUDIO_DIR
# transcription_folder = TRANSCRIPTION_DIR
transcription_folder = TRANSCRIPTION_DIR
# log_file for transcription progress
log_file = os.path.join(DOWNLOAD_LOG_DIR, 'progresso_transcricao.log')

In [ ]:
df = pd.read_csv(os.path.join(CLEANED_DATA_DIR, 'final_comments_with_sentiment.csv'))

In [ ]:
df.shape

In [ ]:
def load_progress():
    if os.path.exists(log_file):
        with open(log_file, "r") as f:
            return set(f.read().splitlines())
    return set()

def save_progress(video_id):
    with open(log_file, "a") as f:
        f.write(video_id + "\n")

In [ ]:
completed_videos = load_progress()

In [ ]:
modelo = whisper.load_model("small").to("cuda")

In [ ]:
for video_id in tqdm(df['video_id'].drop_duplicates(), desc="Transcrevendo áudios"):
    audio_path = os.path.join(audio_folder, f"{video_id}.mp3")
    output_text = os.path.join(transcription_folder, f"{video_id}.txt")
    zip_audio_base = os.path.join(audio_folder, video_id)  # Sem extensão

    # If already transcribed, skip
    if video_id in completed_videos or os.path.exists(output_text):
        print(f"Transcription already done for {video_id}")
        continue

    if not os.path.exists(audio_path):
        print(f"Audio not found for {video_id}, skipping...")
        continue

    print(f"Transcribing {audio_path}...")

    try:
        result = modelo.transcribe(audio_path, language="portuguese")

        with open(output_text, "w", encoding="utf-8") as f:
            f.write(result["text"])

        save_progress(video_id)
        print(f"Transcription saved: {output_text}")

        # Compress only the MP3 file
        shutil.make_archive(zip_audio_base, 'zip', audio_folder, f"{video_id}.mp3")
        print(f"Audio compressed: {zip_audio_base}.zip")
        
        os.remove(audio_path)
        print(f"Audio removed: {audio_path}")

    except Exception as e:
        print(f"Error transcribing {video_id}: {str(e)}")

print("Transcription process finished!")